In [ ]:
import marimo as mo
import cv2
import matplotlib.pyplot as plt
import math

In [ ]:
img = cv2.imread('a4/img.png', cv2.IMREAD_GRAYSCALE)
h, w = img.shape
print(h,w)

In [ ]:
## output frame
## resusable
def output_img_frame(img,output_dim=h-1,f_size=2):
    stride = 1
    output_img_frame = []
    for i in range(output_dim):
        row = []
        for j in range(output_dim):
            row.append(0)
        output_img_frame.append(row)
    return output_img_frame

output_img_frame_frame=output_img_frame(img)

## Robert Detection

In [ ]:
##Robert
def robert(reg):
    p_00 = reg[0][0]
    p_01 = reg[0][1]
    p_10 = reg[1][0]
    p_11 = reg[1][1]
    g_x = p_00 - p_11
    g_y = p_01 - p_10
    mag = math.sqrt(g_x**2 + g_y**2)
    return (int(mag))

## Robert detect
def robert_detect(img, output_img_frame):
    pixels = img.tolist()
    stride = 1
    f_size = 2  # Filter size
    output_dim = 306
    for r in range(output_dim):
        for c in range(output_dim):
     # Determine where the window starts on the big image
            reg = []
            start_y = r * stride
            start_x = c * stride
            # Loop through the filter window
            for ky in range(f_size):
                row = []
                for kx in range(f_size):
                    # Get the pixel value
                    i=(start_y + ky)
                    j=(start_x + kx)
                    val = pixels[i][j]
                    row.append(val)
                reg.append(row)
            ## edge calc
            edge_value = robert(reg)
            output_img_frame[r][c] = edge_value
    return output_img_frame

In [ ]:
empty_out_robert = output_img_frame(img)
robert_detect_out = robert_detect(img, empty_out_robert)

plt.imshow(robert_detect_out, cmap='gray')
plt.gca()

## Prewitt Detection

In [ ]:
def prewitt(reg):
    g_x = ((reg[0][2] + reg[1][2] + reg[2][2])-
    (reg[0][0] + reg[1][0] +reg[2][0]))

    g_y = ((reg[2][0] + reg[2][1] + reg[2][2])-
    (reg[0][0] + reg[0][1] + reg[0][2]))

    mag = math.sqrt(g_x**2 + g_y**2)
    return(int(mag))

In [ ]:
## Robert detect
def main_detect(img, output_img_frame,filter):
    if hasattr(img, 'tolist'):
        pixels = img.tolist()
    else:
        pixels = img
    stride = 1
    f_size = 3  # Filter size
    output_dim = 305 # 307-2
    for r in range(output_dim):
        for c in range(output_dim):
     # Determine where the window starts on the big image
            reg = []
            start_y = r * stride
            start_x = c * stride
            # Loop through the filter window
            for ky in range(f_size):
                row = []
                for kx in range(f_size):
                    # Get the pixel value
                    i=(start_y + ky)
                    j=(start_x + kx)
                    val = pixels[i][j]
                    row.append(val)
                reg.append(row)
            ## edge calc
            edge_value = filter(reg)
            output_img_frame[r][c] = edge_value
    return output_img_frame

## Sobel detect

In [ ]:
## Sobel 
def sobel(reg):
    g_x = ((reg[0][2] + (2 * reg[1][2]) + reg[2][2])-
    (reg[0][0] + (2 * reg[1][0]) + reg[2][0]))
    g_y = ((reg[2][0] + (2 * reg[2][1]) + reg[2][2])-
    (reg[0][0] + (2 * reg[0][1]) + reg[0][2]))

    mag = math.sqrt(g_x**2+g_y**2)
    return int(mag)

In [ ]:
empty_out_prewitt = output_img_frame(img, output_dim=305, f_size=3)

prewitt_detect_out = main_detect(img, empty_out_prewitt,sobel)

plt.imshow(prewitt_detect_out, cmap='gray')
plt.gca()

In [ ]:
empty_out_sobel = output_img_frame(img, output_dim=305, f_size=3)

## we can reuse prewitt_detect due to same logic
sobel_detect_out = main_detect(img, empty_out_sobel,sobel)

plt.imshow(sobel_detect_out, cmap='gray')
plt.gca()

## Laplacian Detection

In [ ]:
def laplacian(reg):
    # Map the kernel directly to the pixels in the 3x3 'reg' patch
    center = -4 * reg[1][1]
    top    = reg[0][1]
    bottom = reg[2][1]
    left   = reg[1][0]
    right  = reg[1][2]

    # total value
    val = center + top + bottom + left + right
    # The Laplacian detects the rate of change, which can result in negative numbers.
    mag = abs(val)
    return int(mag)

In [ ]:
empty_out_laplacian = output_img_frame(img, output_dim=305, f_size=3)

## we can reuse prewitt_detect due to same logic
laplacian_detect_out = main_detect(img, empty_out_laplacian,laplacian)

plt.imshow(laplacian_detect_out, cmap='gray')
plt.gca()

## Gaussian -> Laplacian

In [ ]:
# convert the input into gaussian output
def gaussian_matrix(size,sigma):
    import math
    radius =size//2
    filter = []
    total=0

    ## matrix, inc zeroth pos at centre
    for x in range(-radius, radius+1): 
        row = []
        for y in range(-radius, radius+1):
            exp = -(x**2 + y**2)/(2*sigma**2) # power
            val = math.exp(exp)
            row.append(val)
            total += val
        filter.append(row)
    # normalize
    for i in range(size):
        for j in range(size):
            filter[i][j] = filter[i][j] / total
    return filter



## GAUSSIAN FILTER
def gaussian_filter(img, output_img_frame, filter):
    pixels = img.tolist()
    stride = 1
    f_size = len(filter)
    output_dim = len(output_img_frame)
    for r in range(output_dim):
        for c in range(output_dim):
            # image window on the big image
            start_y = r * stride
            start_x = c * stride
            weighted_sum = 0
            # each pixels in filter
            for ky in range(f_size):
                for kx in range(f_size):
                    # Get the pixel value
                    i=(start_y + ky)
                    j=(start_x + kx)
                    val = pixels[i][j]
                    # apply filter, gives output weight depending on the ith and j th pos of pixel inside filter
                    wt = filter[ky][kx]
                    weighted_sum +=(val*wt)
            output_img_frame[r][c] = int(weighted_sum)
    return output_img_frame

## Running the filter
# Image_store_dimension
empty_out_gaussian = output_img_frame(img, output_dim=303,f_size=5)
## create the gaussian matrix for out specific filter size and sigma
gaussian_matrix = gaussian_matrix(5,2)

gaussian_filtered_img = gaussian_filter(img, empty_out_gaussian, gaussian_matrix)

###
plt.imshow(gaussian_filtered_img, cmap='gray')
plt.gca()

In [ ]:

## Apply Laplacian on this
# output dimension should be nrow-frow/stride +1 = 307-5+1 =303
#Laplacian filter is 3x3
#so 303 - 3+1 = 301
def combined_detect(img, output_img, func):
    # Safe list conversion for the transition from cv2 -> gaussian -> laplacian  
    pixels = img    
    stride = 1
    f_size = 3 
    output_dim = 301

    for r in range(output_dim):
        for c in range(output_dim):
            start_y = r * stride
            start_x = c * stride
# filter region
            reg = []
            for ky in range(f_size):
                row = []
                for kx in range(f_size):
                    i = (start_y + ky)
                    j = (start_x + kx)
                    val = pixels[i][j]  # Your existing syntax
                    row.append(val)
                reg.append(row)
            output_img[r][c] = func(reg)
    return output_img

empty_out_com = output_img_frame(img, output_dim=301, f_size=3)
combined_detect_out = combined_detect(gaussian_filtered_img, empty_out_com, laplacian)

plt.imshow(combined_detect_out, cmap='gray')
plt.gca()

## Canny's edge Detection

In [ ]:
# gaussian_filtered_img
plt.imshow(gaussian_filtered_img, cmap='gray')
plt.gca()
# current output dim = 303

In [ ]:
def canny_sobel(reg):
    g_x=((reg[0][2] + (2 * reg[1][2]) + reg[2][2]) -
           (reg[0][0] + (2 * reg[1][0]) + reg[2][0]))
    g_y=((reg[2][0] + (2 * reg[2][1]) + reg[2][2]) -
           (reg[0][0] + (2 * reg[0][1]) + reg[0][2]))
    mag=math.sqrt(g_x**2 + g_y**2)
    rad=math.atan2(g_y, g_x)
    angle=rad*180/math.pi # convert to theta
    return int(mag), angle

In [ ]:
def canny_sobel_detect(gaussian_img, output_dim):
    mag_img=[]
    angle_img=[]
    # zero matrix
    for i in range(output_dim):
        mag_row=[]
        angle_row=[]
        for j in range(output_dim):
            mag_row.append(0)
            angle_row.append(0)
        mag_img.append(mag_row)
        angle_img.append(angle_row)

    for r in range(1, output_dim-1):
        for c in range(1, output_dim-1):
            #region of interest
            region = [
                [gaussian_img[r-1][c-1], gaussian_img[r-1][c],gaussian_img[r-1][c+1]],
                [gaussian_img[r][c-1],   gaussian_img[r][c],gaussian_img[r][c+1]],
                [gaussian_img[r+1][c-1], gaussian_img[r+1][c], gaussian_img[r+1][c+1]]
            ]
            mag, angle=canny_sobel(region)
            # creating mag and angle matrix for output
            mag_img[r][c]=mag
            angle_img[r][c]=angle
    return mag_img, angle_img

In [ ]:
## NMS apply, thinning
def canny_nms(mag_img, angle_img, output_dim, t_low, t_high
             , weak_val=50, max_val=255):
    nms_out=[]
    for i in range(output_dim):
        nms_out.append([0]*output_dim)
    # putting range 1 to out-1, cuz our region has r-1 and c-1
    for r in range(1, output_dim-1):
        for c in range(1, output_dim-1):
            mag = mag_img[r][c]
            angle= angle_img[r][c]
            ## categorizing
            if angle<0:
                angle+=360
            hor_shift=0
            ver_shift=0

            if (0 <= angle <22.5) or\
            (157.5<=angle<=202.5) or\
            (337.5<=angle<=360):
                hor_shift,ver_shift=0,1
            elif\
            (22.5<=angle<67.5) or\
            (202.5<=angle<247.5):
                hor_shift,ver_shift=-1,1

            elif\
            (67.5<=angle<112.5) or\
            (247.5<=angle<292.5):
                hor_shift,ver_shift=1,0

            elif\
            (112.5<=angle<157.5) or\
            (292.5<=angle<337.5):
                hor_shift,ver_shift=-1,-1
            ## finding neighbours
            ne1=mag_img[r+hor_shift][c+ver_shift]
            ne2=mag_img[r-hor_shift][c-ver_shift]
            #keep pixel if > both nei on that line.

            ## double thresholding
            if mag>=ne1 and mag>=ne2:
                if mag >= t_high:
                    nms_out[r][c]=max_val
                elif mag >= t_low:
                    nms_out[r][c]=weak_val
                else:
                    nms_out[r][c]=0
            else:
                nms_out[r][c]=0
    return nms_out

In [ ]:
mag_img, angle_img = canny_sobel_detect(gaussian_filtered_img, 301)
thresholded_img = canny_nms(mag_img, angle_img, 301, t_low=20, t_high=140, weak_val=50, max_val=255)

plt.imshow(thresholded_img, cmap='gray')
plt.gca()

In [ ]:
## hysteresis
def canny_hysteresis(thresh_img, output_dim, weak_val=50,max_val=255):
    final_out=[]
    for i in range(output_dim):
        final_out.append([0]*output_dim)
    for r in range(1, output_dim-1):
        for c in range(1,output_dim-1):
            ## don't touch already max points/edges
            if thresh_img[r][c]==max_val:
                final_out[r][c]=max_val
            elif thresh_img[r][c] == weak_val:
                ## check filter around the pixel
                if (thresh_img[r-1][c-1]== max_val or
                    thresh_img[r-1][c]  == max_val or
                    thresh_img[r-1][c+1]== max_val or
                    thresh_img[r][c-1]  == max_val or
                    thresh_img[r][c+1]  == max_val or
                    thresh_img[r+1][c-1]== max_val or
                    thresh_img[r+1][c]  == max_val or
                    thresh_img[r+1][c+1]== max_val):
                    final_out[r][c] = max_val
                else:
                    final_out[r][c]=0
            else:
                final_out[r][c] =0
    return final_out

In [ ]:
canny_edges = canny_hysteresis(thresholded_img, 301, weak_val=50, max_val=255)

plt.imshow(canny_edges, cmap='gray')
plt.gca()